# GE to Snowflake MCP Quickstart

This notebook is the shortest path to set up and demo a Gemini Enterprise Custom MCP connection to Snowflake-managed MCP.

What this notebook does:
- validates the semantic view
- creates the OAuth integration
- creates the Snowflake MCP server
- prints the values needed for the GE form
- shows the minimum test path

> Optional: If your account does not enforce restrictive network policy on OAuth token requests, you do not need the temporary allow-all network policy shown later.

> Current project note: In this account, the first successful GE auth happened only after temporarily attaching an allow-all integration-level network policy. Treat that as a diagnostic step, not a default production requirement.

In [ ]:
%%sql -r semantic_view_definition
DESCRIBE SEMANTIC VIEW POC.AI.WEATHER_MAN;

> Required: This notebook assumes the semantic view already exists and is ready for Cortex Analyst.

> Optional: If the customer already has a production semantic view and role model, reuse those instead of the example names in this notebook.

In [ ]:
%%sql -r oauth_integration_create_result
CREATE OR REPLACE SECURITY INTEGRATION GE_MCP_OAUTH_INTEGRATION
  TYPE = OAUTH
  OAUTH_CLIENT = CUSTOM
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = 'https://vertexaisearch.cloud.google.com/oauth-redirect'
  OAUTH_ALLOW_NON_TLS_REDIRECT_URI = TRUE
  OAUTH_ISSUE_REFRESH_TOKENS = TRUE
  OAUTH_REFRESH_TOKEN_VALIDITY = 86400
  OAUTH_ENFORCE_PKCE = FALSE
  ENABLED = TRUE;

In [ ]:
%%sql -r oauth_integration_description
DESCRIBE SECURITY INTEGRATION GE_MCP_OAUTH_INTEGRATION;

In [ ]:
SHOW MCP SERVERS IN SCHEMA POC.AI;

In [ ]:
DESCRIBE MCP SERVER POC.AI.GE_WEATHER_MCP_SERVER;

In [ ]:
%%sql -r oauth_client_secrets
SELECT SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('GE_MCP_OAUTH_INTEGRATION') AS oauth_client_secrets;

# GE Form Values

Use these values in Gemini Enterprise Custom MCP Server setup.

MCP server description:

```text
Snowflake-managed MCP server that lets Gemini Enterprise query Snowflake weather analytics through Cortex Analyst. It exposes the WEATHER_MAN semantic view as an MCP tool so users can ask natural-language questions about NYC weather, temperature, precipitation, snowfall, and wind by date or date range.
```

MCP agent instructions:

```text
Use this MCP server to answer questions about NYC weather data stored in Snowflake.

When responding:
- Use the available weather analyst tool for questions about temperature, precipitation, snowfall, wind, and date-based weather patterns.
- Interpret user questions as analytics requests over the WEATHER_MAN semantic view in Snowflake.
- Prefer direct factual answers first, then include brief supporting context when useful.
- If the user asks for a date, metric, comparison, trend, maximum, minimum, or summary about NYC weather, call the MCP tool.
- If the question is ambiguous, ask a short clarifying question before using the tool.
- If the tool returns no result, say that the available Snowflake weather dataset could not answer the question.
- Do not invent values, dates, or calculations that are not returned by the tool.
- Keep answers concise and business-friendly.
```

Primary MCP endpoint:

```text
https://qn43380.us-central1.gcp.snowflakecomputing.com/api/v2/databases/poc/schemas/ai/mcp-servers/GE_WEATHER_MCP_SERVER
```

> Optional: If GE does not work with the primary endpoint, test this variant instead:

```text
https://qn43380.us-central1.gcp.snowflakecomputing.com/api/v2/databases/poc/schemas/ai/mcp-servers/GE_WEATHER_MCP_SERVER/sse
```

OAuth values:
- Authorization URL: `https://qn43380.us-central1.gcp.snowflakecomputing.com/oauth/authorize`
- Token URL: `https://qn43380.us-central1.gcp.snowflakecomputing.com/oauth/token-request`
- Scope: `refresh_token session:role:POC`
- Redirect URI: `https://vertexaisearch.cloud.google.com/oauth-redirect`

Read the client ID and client secret from the previous SQL cell output.

# Optional Network Policy Diagnostic

> Optional: Run the next three cells only if `Verify Auth` in GE fails.

> POC vs production: the allow-all policy below is only to prove whether network policy is blocking the OAuth token request path. If it fixes auth, replace it later with the narrower policy your security team approves.

In [ ]:
%%sql -r temp_network_policy_create_result
CREATE OR REPLACE NETWORK POLICY TEMP_GE_OAUTH_ALLOW_ALL
  ALLOWED_IP_LIST = ('0.0.0.0/0');

In [ ]:
%%sql -r oauth_integration_policy_update_result
ALTER SECURITY INTEGRATION GE_MCP_OAUTH_INTEGRATION
  SET NETWORK_POLICY = TEMP_GE_OAUTH_ALLOW_ALL;

In [ ]:
%%sql -r oauth_integration_after_policy_change
DESCRIBE SECURITY INTEGRATION GE_MCP_OAUTH_INTEGRATION;

# Next Step: Test in Gemini Enterprise

After the GE connection is created, test the simplest possible path:

1. Open the GE experience where the MCP connector is enabled.
2. Confirm the connector/auth looks healthy.
3. Ask one dataset-specific question.

Suggested prompts:

```text
What was the hottest day in NYC in 2021?
```

```text
What was the average maximum temperature in March 2021?
```

```text
How much snowfall was recorded in January 2021?
```

Success criteria:
- GE stays on the custom MCP path instead of giving a generic answer
- the answer is grounded in the weather dataset

If auth is green but the chat still does not use the connector, the next troubleshooting layer is GE runtime tool discovery, not Snowflake OAuth.

# Cleanup

> Optional: If you used the temporary allow-all policy only to diagnose auth, remove it after deciding the final production posture.

```sql
ALTER SECURITY INTEGRATION GE_MCP_OAUTH_INTEGRATION
  UNSET NETWORK_POLICY;

DROP NETWORK POLICY IF EXISTS TEMP_GE_OAUTH_ALLOW_ALL;
```